In [ ]:
### CELL 1 - GOOGLE DRIVE SETUP ###
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
### CELL 2 - DATA SANITY CHECK FOR V50 ###
# Comprehensive analysis of parquet race data:
#   1. Feature distributions & scale audit
#   2. Feature-price movement correlations
#   3. Cross-runner dynamics (overround compression, coupled movements)
#   4. Feature stationarity (early vs late market)
#   5. Tradeable opportunity sizing
#   6. Data quality summary
#
# Mirrors the exact feature engineering in V50a MarketMakingEnv

import os
import numpy as np
import pandas as pd
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/content/drive/MyDrive/race_out'

# ============================================================
# LOAD ALL RACE FILES
# ============================================================
print('=' * 70)
print('  DATA SANITY CHECK — V50 Feature Analysis')
print('=' * 70)

all_files = sorted([os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR)
                    if f.endswith('.parquet')])
print(f'\nFound {len(all_files)} parquet files in {DATA_DIR}')

# Load a sample to get schema
sample_df = pd.read_parquet(all_files[0])
n_cols = len(sample_df.columns)
runner_cols_sample = [c for c in sample_df.columns if c.startswith('run[0].')]
print(f'Schema: {n_cols} columns, {len(runner_cols_sample)} per-runner features')
print(f'Global columns: {n_cols - len(runner_cols_sample) * int(sample_df.iloc[0]["runner_count"])}')


# ============================================================
# HELPER: replicate V50a feature engineering for a single row
# ============================================================
def safe_price_norm(price, epsilon=1e-8):
    if price < 1.01:
        return 0.0
    return float(np.log(max(price, 1.01) + epsilon) / np.log(1001.0))

def safe_log_norm(value, epsilon=1e-8):
    return float(np.log1p(max(0.0, float(value)) + epsilon))

def extract_runner_features(row, runner_idx, runner_count):
    """Extract raw + engineered features for one runner, matching V50a."""
    prefix = f'run[{runner_idx}].'
    bp1 = row.get(f'{prefix}back_price_1', np.nan)
    lp1 = row.get(f'{prefix}lay_price_1', np.nan)
    if pd.isna(bp1) or pd.isna(lp1):
        return None

    bv1 = float(row.get(f'{prefix}back_size_1', 0) or 0)
    lv1 = float(row.get(f'{prefix}lay_size_1', 0) or 0)
    bv2 = float(row.get(f'{prefix}back_size_2', 0) or 0)
    lv2 = float(row.get(f'{prefix}lay_size_2', 0) or 0)
    bv3 = float(row.get(f'{prefix}back_size_3', 0) or 0)
    lv3 = float(row.get(f'{prefix}lay_size_3', 0) or 0)

    microprice = row.get(f'{prefix}microprice', np.nan)
    if pd.isna(microprice):
        microprice = (bp1 + lp1) / 2.0

    ob_imb = row.get(f'{prefix}ob_imbalance', 0.0)
    if pd.isna(ob_imb):
        total_v = bv1 + lv1
        ob_imb = ((bv1 - lv1) / total_v) if total_v > 0 else 0.0

    rel_spread = row.get(f'{prefix}rel_spread', np.nan)
    if pd.isna(rel_spread):
        rel_spread = abs(bp1 - lp1) / max(microprice, 1.01)

    prob_implied = row.get(f'{prefix}prob_implied', np.nan)
    if pd.isna(prob_implied):
        prob_implied = 1.0 / microprice if microprice > 1.01 else 0.5

    traded_vol_total = float(row.get(f'{prefix}traded_vol_total', 0) or 0)
    traded_vol_60s = float(row.get(f'{prefix}traded_vol_60s', 0) or 0)
    secs_since_trade = float(row.get(f'{prefix}secs_since_last_trade', 999) or 999)
    ret_std_5s = float(row.get(f'{prefix}ret_std_5s', 0) or 0)
    ret_std_20s = float(row.get(f'{prefix}ret_std_20s', 0) or 0)

    total_back_liq = bv1 + bv2 + bv3
    total_lay_liq = lv1 + lv2 + lv3
    depth_conc = (bv1 / total_back_liq) if total_back_liq > 0 else 1.0
    vol_accel = (traded_vol_60s / traded_vol_total) if traded_vol_total > 0 else 0.0

    return {
        'microprice': float(microprice),
        'back_1': float(bp1),
        'lay_1': float(lp1),
        'back_vol_1': bv1,
        'lay_vol_1': lv1,
        'ob_imbalance': float(ob_imb),
        'rel_spread': float(rel_spread),
        'prob_implied': float(prob_implied),
        'traded_vol_total': traded_vol_total,
        'traded_vol_60s': traded_vol_60s,
        'secs_since_last_trade': secs_since_trade,
        'ret_std_5s': ret_std_5s,
        'ret_std_20s': ret_std_20s,
        'total_back_liq': total_back_liq,
        'total_lay_liq': total_lay_liq,
        'depth_concentration': depth_conc,
        'vol_acceleration': vol_accel,
        # Normalized versions (as fed to network)
        'microprice_norm': safe_price_norm(microprice),
        'back_vol_1_norm': safe_log_norm(bv1),
        'lay_vol_1_norm': safe_log_norm(lv1),
        'traded_vol_total_norm': safe_log_norm(traded_vol_total),
        'traded_vol_60s_norm': safe_log_norm(traded_vol_60s),
        'ret_std_5s_norm': safe_log_norm(ret_std_5s),
    }

print('\nHelpers loaded. Processing races...')


In [ ]:
### CELL 3 - PROCESS ALL RACES ###
# Collect per-step feature snapshots and price movements across all races

MAX_RACES = 2000  # cap for memory; set higher if needed

# Accumulators
all_feature_rows = []       # per-runner-per-step feature dicts
all_global_rows = []        # per-step global feature dicts
race_summaries = []         # per-race summary stats
cross_runner_rows = []      # per-step cross-runner snapshots

loaded = 0
failed = 0

for fi, filepath in enumerate(all_files[:MAX_RACES]):
    try:
        df = pd.read_parquet(filepath)
    except Exception:
        failed += 1
        continue

    if len(df) < 10 or 'run[0].back_price_1' not in df.columns:
        failed += 1
        continue

    runner_count = int(df.iloc[0].get('runner_count', 0))
    if runner_count < 2:
        failed += 1
        continue

    # Split pre-race only
    if 'in_play' in df.columns:
        pre = df[df['in_play'] == 0].copy()
    else:
        pre = df.copy()

    if len(pre) < 5:
        failed += 1
        continue

    loaded += 1

    # --- Per-race summary ---
    race_info = {
        'file': os.path.basename(filepath),
        'runner_count': runner_count,
        'pre_race_steps': len(pre),
        'total_steps': len(df),
        'in_play_steps': len(df) - len(pre),
    }

    # Secs to off
    if 'secs_to_off' in pre.columns:
        race_info['secs_to_off_start'] = pre['secs_to_off'].iloc[0]
        race_info['secs_to_off_end'] = pre['secs_to_off'].iloc[-1]
        race_info['pre_race_duration'] = abs(pre['secs_to_off'].iloc[0] - pre['secs_to_off'].iloc[-1])

    # Overround
    if 'back_overround' in pre.columns:
        race_info['overround_start'] = pre['back_overround'].iloc[0]
        race_info['overround_end'] = pre['back_overround'].iloc[-1]
        race_info['overround_compression'] = race_info['overround_start'] - race_info['overround_end']

    if 'total_matched_market' in pre.columns:
        race_info['matched_at_off'] = pre['total_matched_market'].iloc[-1]

    # Per-runner price ranges
    runner_ranges = []
    for ri in range(min(runner_count, 24)):
        mp_col = f'run[{ri}].microprice'
        if mp_col in pre.columns:
            mp = pre[mp_col].dropna()
            if len(mp) > 1:
                mp_range_pct = (mp.max() - mp.min()) / mp.iloc[0] * 100 if mp.iloc[0] > 1 else 0
                runner_ranges.append(mp_range_pct)

    race_info['mean_runner_range_pct'] = np.mean(runner_ranges) if runner_ranges else 0
    race_info['max_runner_range_pct'] = np.max(runner_ranges) if runner_ranges else 0
    race_info['fav_range_pct'] = runner_ranges[0] if runner_ranges else 0
    race_summaries.append(race_info)

    # --- Per-step features (subsample for memory) ---
    step_indices = range(0, len(pre) - 1)  # -1 because we need next step for target
    for si in step_indices:
        row_now = pre.iloc[si]
        row_next = pre.iloc[si + 1]

        # Global features
        g = {
            'back_overround': row_now.get('back_overround', np.nan),
            'lay_overround': row_now.get('lay_overround', np.nan),
            'overround_gap': row_now.get('overround_gap', np.nan),
            'prob_entropy': row_now.get('prob_entropy', np.nan),
            'total_matched_market': row_now.get('total_matched_market', np.nan),
            'secs_to_off': row_now.get('secs_to_off', np.nan),
            'spread_mean_run': row_now.get('spread_mean_run', np.nan),
            'fav_prob_gap': row_now.get('fav_prob_gap', np.nan),
        }
        all_global_rows.append(g)

        # Cross-runner snapshot for this step
        step_implied_probs = []
        step_volumes = []
        step_microprices_now = []
        step_microprices_next = []

        for ri in range(min(runner_count, 24)):
            feat_now = extract_runner_features(row_now, ri, runner_count)
            feat_next = extract_runner_features(row_next, ri, runner_count)
            if feat_now is None or feat_next is None:
                continue

            # Price movement target: next-step microprice return
            if feat_now['microprice'] > 1.01 and feat_next['microprice'] > 1.01:
                price_return = (feat_next['microprice'] - feat_now['microprice']) / feat_now['microprice']
            else:
                price_return = 0.0

            feat_now['price_return_1step'] = price_return
            feat_now['race_idx'] = loaded
            feat_now['step_idx'] = si
            feat_now['runner_idx'] = ri
            feat_now['secs_to_off'] = row_now.get('secs_to_off', np.nan)
            all_feature_rows.append(feat_now)

            step_implied_probs.append(feat_now['prob_implied'])
            step_volumes.append(feat_now['traded_vol_total'])
            step_microprices_now.append(feat_now['microprice'])
            if feat_next:
                step_microprices_next.append(feat_next['microprice'])

        # Cross-runner snapshot
        if len(step_implied_probs) >= 2 and len(step_microprices_next) == len(step_microprices_now):
            total_imp = sum(step_implied_probs)
            total_vol = sum(step_volumes) if sum(step_volumes) > 0 else 1.0
            cross_runner_rows.append({
                'n_runners': len(step_implied_probs),
                'sum_implied_prob': total_imp,
                'back_overround': row_now.get('back_overround', np.nan),
                'secs_to_off': row_now.get('secs_to_off', np.nan),
                'prob_shares': [p / total_imp for p in step_implied_probs],
                'vol_shares': [v / total_vol for v in step_volumes],
                'microprices_now': step_microprices_now,
                'microprices_next': step_microprices_next,
            })

    if loaded % 200 == 0:
        print(f'  Processed {loaded} races ({fi+1}/{len(all_files[:MAX_RACES])} files)...')

print(f'\nDone: {loaded} races loaded, {failed} failed/skipped')
print(f'Feature rows: {len(all_feature_rows):,}')
print(f'Global rows: {len(all_global_rows):,}')
print(f'Cross-runner rows: {len(cross_runner_rows):,}')

# Convert to DataFrames
df_feat = pd.DataFrame(all_feature_rows)
df_global = pd.DataFrame(all_global_rows)
df_races = pd.DataFrame(race_summaries)
print(f'\nDataFrames built: features={df_feat.shape}, global={df_global.shape}, races={df_races.shape}')


In [ ]:
### CELL 4 - SECTION 1: FEATURE DISTRIBUTIONS & SCALE AUDIT ###
print('=' * 70)
print('  SECTION 1: Feature Distributions & Scale Audit')
print('  (Are the normalization choices in V50a appropriate?)')
print('=' * 70)

# Raw feature distributions
raw_features = [
    'microprice', 'back_1', 'lay_1', 'back_vol_1', 'lay_vol_1',
    'ob_imbalance', 'rel_spread', 'prob_implied',
    'traded_vol_total', 'traded_vol_60s', 'secs_since_last_trade',
    'ret_std_5s', 'ret_std_20s', 'total_back_liq', 'total_lay_liq',
    'depth_concentration', 'vol_acceleration',
]

print(f'\n{"Feature":<25s} {"Mean":>10s} {"Std":>10s} {"Min":>10s} {"p5":>10s} {"p50":>10s} {"p95":>10s} {"Max":>10s} {"Nulls":>6s}')
print('-' * 107)
for feat in raw_features:
    if feat not in df_feat.columns:
        continue
    s = df_feat[feat].dropna()
    if len(s) == 0:
        continue
    print(f'{feat:<25s} {s.mean():10.4f} {s.std():10.4f} {s.min():10.4f} '
          f'{s.quantile(0.05):10.4f} {s.quantile(0.5):10.4f} {s.quantile(0.95):10.4f} '
          f'{s.max():10.4f} {df_feat[feat].isna().sum():6d}')

# Normalized feature distributions (as the network sees them)
print(f'\n\nNormalized features (as fed to SAC network):')
print(f'{"Feature":<25s} {"Mean":>10s} {"Std":>10s} {"Min":>10s} {"p5":>10s} {"p50":>10s} {"p95":>10s} {"Max":>10s}')
print('-' * 95)
norm_features = [
    'microprice_norm', 'back_vol_1_norm', 'lay_vol_1_norm',
    'traded_vol_total_norm', 'traded_vol_60s_norm', 'ret_std_5s_norm',
]
for feat in norm_features:
    if feat not in df_feat.columns:
        continue
    s = df_feat[feat].dropna()
    if len(s) == 0:
        continue
    print(f'{feat:<25s} {s.mean():10.4f} {s.std():10.4f} {s.min():10.4f} '
          f'{s.quantile(0.05):10.4f} {s.quantile(0.5):10.4f} {s.quantile(0.95):10.4f} '
          f'{s.max():10.4f}')

# Scale mismatch analysis
print(f'\n\nSCALE MISMATCH ANALYSIS:')
print(f'  The SAC critic sees all features concatenated. If scales differ')
print(f'  by >10x, the critic will struggle to learn Q-values.')
scale_features = {
    'microprice_norm': df_feat['microprice_norm'].std() if 'microprice_norm' in df_feat else 0,
    'back_vol_1_norm': df_feat['back_vol_1_norm'].std() if 'back_vol_1_norm' in df_feat else 0,
    'ob_imbalance': df_feat['ob_imbalance'].std() if 'ob_imbalance' in df_feat else 0,
    'rel_spread': df_feat['rel_spread'].std() if 'rel_spread' in df_feat else 0,
    'depth_concentration': df_feat['depth_concentration'].std() if 'depth_concentration' in df_feat else 0,
    'vol_acceleration': df_feat['vol_acceleration'].std() if 'vol_acceleration' in df_feat else 0,
    'ret_std_5s_norm': df_feat['ret_std_5s_norm'].std() if 'ret_std_5s_norm' in df_feat else 0,
}
max_scale = max(scale_features.values())
min_scale = min(v for v in scale_features.values() if v > 0)
print(f'  Scale range: {min_scale:.4f} to {max_scale:.4f} (ratio: {max_scale/max(min_scale, 1e-8):.1f}x)')
for name, std in sorted(scale_features.items(), key=lambda x: -x[1]):
    bar = '#' * int(std / max_scale * 40)
    print(f'    {name:<25s} std={std:.4f}  {bar}')

if max_scale / max(min_scale, 1e-8) > 10:
    print(f'\n  WARNING: Scale mismatch > 10x. VecNormalize (Phase V50b) will help.')
else:
    print(f'\n  Scales are within 10x — manageable without VecNormalize.')


In [ ]:
### CELL 5 - SECTION 2: FEATURE-PRICE MOVEMENT CORRELATIONS ###
print('=' * 70)
print('  SECTION 2: Feature — Price Movement Correlations')
print('  (Which features predict next-step microprice returns?)')
print('=' * 70)

corr_features = [
    'ob_imbalance', 'rel_spread', 'prob_implied',
    'traded_vol_60s', 'vol_acceleration', 'depth_concentration',
    'ret_std_5s', 'ret_std_20s', 'secs_since_last_trade',
    'total_back_liq', 'total_lay_liq',
    'back_vol_1', 'lay_vol_1',
    'microprice_norm', 'traded_vol_total_norm',
]

target = 'price_return_1step'
if target in df_feat.columns:
    print(f'\nTarget: {target}')
    print(f'  Mean: {df_feat[target].mean():.6f}')
    print(f'  Std:  {df_feat[target].std():.6f}')
    print(f'  Abs mean: {df_feat[target].abs().mean():.6f}')
    print(f'  Non-zero: {(df_feat[target] != 0).sum():,} / {len(df_feat):,}')

    print(f'\n{"Feature":<25s} {"Pearson r":>10s} {"Rank corr":>10s} {"Abs corr":>10s}  Direction')
    print('-' * 75)

    results = []
    for feat in corr_features:
        if feat not in df_feat.columns:
            continue
        valid = df_feat[[feat, target]].dropna()
        if len(valid) < 100:
            continue
        pearson = valid[feat].corr(valid[target])
        spearman = valid[feat].corr(valid[target], method='spearman')
        direction = 'shortens' if pearson < 0 else 'drifts out' if pearson > 0 else 'neutral'
        results.append((feat, pearson, spearman, abs(pearson), direction))

    results.sort(key=lambda x: -x[3])
    for feat, pearson, spearman, abs_corr, direction in results:
        bar = '#' * int(abs_corr * 500)  # scale for visibility
        print(f'{feat:<25s} {pearson:10.4f} {spearman:10.4f} {abs_corr:10.4f}  {direction}  {bar}')

    # Interpretation
    print(f'\nINTERPRETATION:')
    top3 = results[:3]
    for feat, pearson, _, _, direction in top3:
        print(f'  {feat}: r={pearson:.4f} — higher {feat} predicts price {direction}')

    if all(abs(r[1]) < 0.01 for r in results):
        print(f'\n  NOTE: All correlations < 0.01. Single-step returns are very noisy.')
        print(f'  This is expected for microstructure data. Signal may emerge over')
        print(f'  multiple steps or conditional on market regime (see Section 4).')
else:
    print('  ERROR: price_return_1step not computed')


In [ ]:
### CELL 6 - SECTION 2b: MULTI-STEP AND CONDITIONAL CORRELATIONS ###
print('=' * 70)
print('  SECTION 2b: Multi-Step Returns & Conditional Correlations')
print('=' * 70)

# Compute 5-step and 10-step returns per runner within each race
print('\nComputing multi-step returns...')
df_feat['price_return_5step'] = np.nan
df_feat['price_return_10step'] = np.nan

for (race_idx, runner_idx), group in df_feat.groupby(['race_idx', 'runner_idx']):
    idx = group.index
    mp = group['microprice'].values
    for offset, col in [(5, 'price_return_5step'), (10, 'price_return_10step')]:
        returns = np.full(len(mp), np.nan)
        for i in range(len(mp) - offset):
            if mp[i] > 1.01 and mp[i + offset] > 1.01:
                returns[i] = (mp[i + offset] - mp[i]) / mp[i]
        df_feat.loc[idx, col] = returns

for horizon, col in [('5-step', 'price_return_5step'), ('10-step', 'price_return_10step')]:
    valid = df_feat[col].dropna()
    print(f'\n{horizon} returns: mean={valid.mean():.6f}, std={valid.std():.6f}, n={len(valid):,}')

    print(f'{"Feature":<25s} {"Pearson r":>10s} {"Abs r":>10s}')
    print('-' * 50)
    results = []
    for feat in corr_features:
        if feat not in df_feat.columns:
            continue
        v = df_feat[[feat, col]].dropna()
        if len(v) < 100:
            continue
        r = v[feat].corr(v[col])
        results.append((feat, r, abs(r)))
    results.sort(key=lambda x: -x[2])
    for feat, r, ar in results[:10]:
        print(f'{feat:<25s} {r:10.4f} {ar:10.4f}')

# Conditional correlations: early market vs late market
print(f'\n\nCONDITIONAL CORRELATIONS (by market phase):')
if 'secs_to_off' in df_feat.columns:
    sto = df_feat['secs_to_off'].dropna()
    # secs_to_off is negative (countdown), so more negative = further from off
    median_sto = sto.median()
    early = df_feat[df_feat['secs_to_off'] < median_sto]  # more negative = earlier
    late = df_feat[df_feat['secs_to_off'] >= median_sto]   # closer to off
    print(f'  Early market: secs_to_off < {median_sto:.0f} ({len(early):,} rows)')
    print(f'  Late market:  secs_to_off >= {median_sto:.0f} ({len(late):,} rows)')

    for label, subset in [('EARLY', early), ('LATE', late)]:
        print(f'\n  {label} market correlations with 5-step return:')
        print(f'  {"Feature":<25s} {"r":>8s}')
        results = []
        for feat in ['ob_imbalance', 'rel_spread', 'vol_acceleration', 'depth_concentration', 'traded_vol_60s']:
            if feat in subset.columns and 'price_return_5step' in subset.columns:
                v = subset[[feat, 'price_return_5step']].dropna()
                if len(v) > 50:
                    r = v[feat].corr(v['price_return_5step'])
                    results.append((feat, r))
        results.sort(key=lambda x: -abs(x[1]))
        for feat, r in results:
            print(f'  {feat:<25s} {r:8.4f}')
else:
    print('  secs_to_off not available')


In [ ]:
### CELL 7 - SECTION 3: CROSS-RUNNER DYNAMICS ###
print('=' * 70)
print('  SECTION 3: Cross-Runner Dynamics')
print('  (Does overround compress? When one shortens, do others drift?)')
print('=' * 70)

# 3a. Overround compression over time
if 'overround_start' in df_races.columns and 'overround_end' in df_races.columns:
    print(f'\nOVERROUND COMPRESSION:')
    print(f'  Start (median): {df_races["overround_start"].median():.1f}%')
    print(f'  End (median):   {df_races["overround_end"].median():.1f}%')
    print(f'  Compression (median): {df_races["overround_compression"].median():.1f}pp')
    print(f'  Compression range: [{df_races["overround_compression"].quantile(0.05):.1f}, {df_races["overround_compression"].quantile(0.95):.1f}]pp')
    print(f'  Races where overround increased: {(df_races["overround_compression"] < 0).sum()} / {len(df_races)}')

# 3b. Cross-runner price correlation
print(f'\nCROSS-RUNNER PRICE MOVEMENT CORRELATION:')
print(f'  When runner A shortens, do others drift out?')

corr_samples = []
for cr in cross_runner_rows[:5000]:  # sample for speed
    mp_now = np.array(cr['microprices_now'])
    mp_next = np.array(cr['microprices_next'])
    if len(mp_now) < 3:
        continue
    returns = (mp_next - mp_now) / np.maximum(mp_now, 1.01)
    # For each runner, correlate its return with mean of OTHER runners' returns
    for i in range(len(returns)):
        others = np.delete(returns, i)
        if len(others) > 0 and np.std(others) > 1e-8:
            corr_samples.append(np.corrcoef(returns[i:i+1], [others.mean()])[0, 1])

if corr_samples:
    cs = np.array(corr_samples)
    cs = cs[~np.isnan(cs)]
    if len(cs) == 0:
        print(f'  Insufficient data for cross-runner correlation (need more races)')
    else:
        print(f'  Mean correlation (runner vs others): {cs.mean():.4f}')
        print(f'  Median: {np.median(cs):.4f}')
        print(f'  % negative (expected if zero-sum): {(cs < 0).mean()*100:.1f}%')
        if cs.mean() < -0.05:
            print(f'  CONFIRMED: negative correlation — when one runner shortens, others tend to drift.')
            print(f'  This validates the overround constraint mechanism.')
        elif cs.mean() > 0.05:
            print(f'  UNEXPECTED: positive correlation — runners move together (market-wide shifts).')
        else:
            print(f'  WEAK: near-zero correlation — runner moves are mostly independent.')

# 3c. Volume concentration vs price movement
print(f'\nVOLUME CONCENTRATION vs PRICE MOVEMENT:')
print(f'  Does disproportionate volume predict subsequent price movement?')

vol_move_corrs = []
for cr in cross_runner_rows[:5000]:
    if len(cr['vol_shares']) < 3 or len(cr['prob_shares']) < 3:
        continue
    vs = np.array(cr['vol_shares'])
    ps = np.array(cr['prob_shares'])
    mp_now = np.array(cr['microprices_now'])
    mp_next = np.array(cr['microprices_next'])
    if len(mp_now) != len(vs):
        continue
    returns = (mp_next - mp_now) / np.maximum(mp_now, 1.01)
    # vol-prob divergence: positive = getting more volume than probability implies
    divergence = vs - ps
    if np.std(divergence) > 1e-8 and np.std(returns) > 1e-8:
        r = np.corrcoef(divergence, returns)[0, 1]
        if not np.isnan(r):
            vol_move_corrs.append(r)

if vol_move_corrs:
    vmc = np.array(vol_move_corrs)
    print(f'  Correlation (vol-prob divergence vs next-step return): {vmc.mean():.4f}')
    print(f'  Median: {np.median(vmc):.4f}')
    if vmc.mean() < -0.02:
        print(f'  SIGNAL: runners receiving disproportionate volume tend to SHORTEN.')
        print(f'  This supports adding vol_share and vol_prob_divergence features.')
    elif vmc.mean() > 0.02:
        print(f'  REVERSE: runners receiving more volume tend to DRIFT OUT.')
    else:
        print(f'  WEAK signal from volume-probability divergence.')


In [ ]:
### CELL 8 - SECTION 4: FEATURE STATIONARITY ###
print('=' * 70)
print('  SECTION 4: Feature Stationarity')
print('  (Do feature distributions change between early and late market?)')
print('=' * 70)

if 'secs_to_off' not in df_feat.columns:
    print('  secs_to_off not available — skipping')
else:
    sto = df_feat['secs_to_off'].dropna()
    q33 = sto.quantile(0.33)
    q66 = sto.quantile(0.66)
    early = df_feat[df_feat['secs_to_off'] < q33]
    mid = df_feat[(df_feat['secs_to_off'] >= q33) & (df_feat['secs_to_off'] < q66)]
    late = df_feat[df_feat['secs_to_off'] >= q66]

    print(f'\n  Early (>{abs(q33):.0f}s out): {len(early):,} rows')
    print(f'  Mid:                        {len(mid):,} rows')
    print(f'  Late (<{abs(q66):.0f}s out):  {len(late):,} rows')

    stat_features = ['ob_imbalance', 'rel_spread', 'vol_acceleration',
                     'depth_concentration', 'traded_vol_60s', 'ret_std_5s']

    print(f'\n  {"Feature":<25s} {"Early mean":>12s} {"Mid mean":>12s} {"Late mean":>12s} {"Early std":>12s} {"Late std":>12s} {"Shift?":>8s}')
    print('  ' + '-' * 95)

    for feat in stat_features:
        if feat not in df_feat.columns:
            continue
        e_mean = early[feat].mean()
        m_mean = mid[feat].mean()
        l_mean = late[feat].mean()
        e_std = early[feat].std()
        l_std = late[feat].std()
        # Check if mean shifted by > 0.5 std
        pooled_std = df_feat[feat].std()
        shift = abs(l_mean - e_mean) / max(pooled_std, 1e-8)
        flag = 'YES' if shift > 0.5 else 'no'
        print(f'  {feat:<25s} {e_mean:12.4f} {m_mean:12.4f} {l_mean:12.4f} {e_std:12.4f} {l_std:12.4f} {flag:>8s}')

    print(f'\n  Features marked YES shift >0.5 std between early and late market.')
    print(f'  These may confuse the critic if not time-conditioned.')
    print(f'  V50a includes secs_to_off as a global feature, which helps.')

    # Overround stationarity
    if 'back_overround' in df_global.columns and 'secs_to_off' in df_global.columns:
        print(f'\n  OVERROUND by market phase:')
        g = df_global.dropna(subset=['back_overround', 'secs_to_off'])
        g_q33 = g['secs_to_off'].quantile(0.33)
        g_q66 = g['secs_to_off'].quantile(0.66)
        for label, mask in [('Early', g['secs_to_off'] < g_q33),
                             ('Mid', (g['secs_to_off'] >= g_q33) & (g['secs_to_off'] < g_q66)),
                             ('Late', g['secs_to_off'] >= g_q66)]:
            sub = g[mask]
            print(f'    {label}: overround mean={sub["back_overround"].mean():.1f}%, '
                  f'median={sub["back_overround"].median():.1f}%')


In [ ]:
### CELL 9 - SECTION 5: TRADEABLE OPPORTUNITY SIZING ###
print('=' * 70)
print('  SECTION 5: Tradeable Opportunity Sizing')
print('  (How much price movement exists per race? What\'s the agent\'s ceiling?)')
print('=' * 70)

if len(df_races) > 0:
    print(f'\nPRICE MOVEMENT PER RACE (% microprice range, pre-race only):')
    print(f'  Favourite runner:')
    s = df_races['fav_range_pct'].dropna()
    print(f'    Mean: {s.mean():.1f}%  |  Median: {s.median():.1f}%  |  p90: {s.quantile(0.90):.1f}%')
    print(f'    Races with >5% fav movement: {(s > 5).sum()} / {len(s)} ({(s>5).mean()*100:.0f}%)')
    print(f'    Races with >10% fav movement: {(s > 10).sum()} / {len(s)} ({(s>10).mean()*100:.0f}%)')

    print(f'\n  All runners (mean across field):')
    s = df_races['mean_runner_range_pct'].dropna()
    print(f'    Mean: {s.mean():.1f}%  |  Median: {s.median():.1f}%  |  p90: {s.quantile(0.90):.1f}%')

    print(f'\n  Max runner range (best opportunity per race):')
    s = df_races['max_runner_range_pct'].dropna()
    print(f'    Mean: {s.mean():.1f}%  |  Median: {s.median():.1f}%  |  p90: {s.quantile(0.90):.1f}%')

    # Estimate theoretical PnL ceiling
    # If agent perfectly timed back-at-low, lay-at-high on the best runner:
    # PnL ≈ stake * (price_range / exit_price)
    # With $100 stake on median best runner (median max range):
    median_max_range = df_races['max_runner_range_pct'].median()
    print(f'\n  THEORETICAL PnL CEILING (perfect timing, $100 stake, best runner):')
    print(f'    Median max range: {median_max_range:.1f}%')
    print(f'    Gross PnL: ~${100 * median_max_range / 100:.2f}')
    print(f'    Net of 5% commission: ~${100 * median_max_range / 100 * 0.95:.2f}')
    print(f'    This is the per-race ceiling — the agent can never beat this.')

    # Pre-race duration
    if 'pre_race_duration' in df_races.columns:
        print(f'\nPRE-RACE WINDOW DURATION (seconds):')
        s = df_races['pre_race_duration'].dropna()
        print(f'  Mean: {s.mean():.0f}s  |  Median: {s.median():.0f}s  |  Range: [{s.min():.0f}s, {s.max():.0f}s]')
        print(f'  Races < 60s: {(s < 60).sum()} / {len(s)}')

    # Matched volume at off
    if 'matched_at_off' in df_races.columns:
        print(f'\nMATCHED VOLUME AT OFF:')
        s = df_races['matched_at_off'].dropna()
        print(f'  Mean: ${s.mean():,.0f}  |  Median: ${s.median():,.0f}')
        print(f'  <$5k: {(s < 5000).sum()} | $5k-50k: {((s >= 5000) & (s < 50000)).sum()} | >$50k: {(s >= 50000).sum()}')

    # Runner count distribution
    print(f'\nRUNNER COUNT DISTRIBUTION:')
    rc = df_races['runner_count']
    for n in sorted(rc.unique()):
        count = (rc == n).sum()
        pct = count / len(rc) * 100
        bar = '#' * int(pct / 2)
        print(f'  {n:2d} runners: {count:4d} races ({pct:5.1f}%) {bar}')


In [ ]:
### CELL 10 - SECTION 6: DATA QUALITY SUMMARY ###
print('=' * 70)
print('  SECTION 6: Data Quality Summary')
print('=' * 70)

print(f'\nDATASET OVERVIEW:')
print(f'  Total parquet files: {len(all_files)}')
print(f'  Successfully loaded: {loaded}')
print(f'  Failed/skipped: {failed}')
print(f'  Load rate: {loaded/(loaded+failed)*100:.1f}%')

# Null rates across key runner features
print(f'\nNULL RATES (across all runner-step rows):')
null_features = ['microprice', 'ob_imbalance', 'rel_spread', 'prob_implied',
                 'ret_std_5s', 'traded_vol_60s', 'secs_since_last_trade']
for feat in null_features:
    if feat in df_feat.columns:
        null_pct = df_feat[feat].isna().mean() * 100
        print(f'  {feat:<25s}: {null_pct:.2f}% null')

# Stale market prevalence
if 'secs_since_last_trade' in df_feat.columns:
    print(f'\nSTALE MARKET PREVALENCE (secs_since_last_trade > 60):')
    stale = (df_feat['secs_since_last_trade'] > 60).mean() * 100
    very_stale = (df_feat['secs_since_last_trade'] > 300).mean() * 100
    print(f'  Stale (>60s): {stale:.1f}% of runner-steps')
    print(f'  Very stale (>300s): {very_stale:.1f}% of runner-steps')
    print(f'  At V50a penalty of -0.02 per stale trade attempt, this is significant.')

# Global feature null rates
print(f'\nGLOBAL FEATURE NULL RATES:')
for feat in df_global.columns:
    null_pct = df_global[feat].isna().mean() * 100
    if null_pct > 0:
        print(f'  {feat:<25s}: {null_pct:.2f}% null')
if all(df_global[f].isna().mean() == 0 for f in df_global.columns):
    print(f'  All global features complete (0% null)')

# Suitability filter summary
print(f'\nSUITABILITY FILTER IMPACT:')
if 'pre_race_duration' in df_races.columns:
    pass_duration = (df_races['pre_race_duration'] >= 60).sum()
    print(f'  Pre-race >= 60s: {pass_duration} / {len(df_races)} ({pass_duration/len(df_races)*100:.0f}%)')
if 'pre_race_steps' in df_races.columns:
    pass_steps = (df_races['pre_race_steps'] >= 30).sum()
    print(f'  Pre-race >= 30 snapshots: {pass_steps} / {len(df_races)} ({pass_steps/len(df_races)*100:.0f}%)')
if 'matched_at_off' in df_races.columns:
    pass_matched = (df_races['matched_at_off'] >= 5000).sum()
    print(f'  Matched >= $5k: {pass_matched} / {len(df_races)} ({pass_matched/len(df_races)*100:.0f}%)')
    pass_matched_15k = (df_races['matched_at_off'] >= 15000).sum()
    print(f'  Matched >= $15k (V50e proposal): {pass_matched_15k} / {len(df_races)} ({pass_matched_15k/len(df_races)*100:.0f}%)')
if 'fav_range_pct' in df_races.columns:
    pass_range = (df_races['fav_range_pct'] >= 1.0).sum()
    print(f'  Fav range >= 1%: {pass_range} / {len(df_races)} ({pass_range/len(df_races)*100:.0f}%)')
    pass_range_3 = (df_races['fav_range_pct'] >= 3.0).sum()
    print(f'  Fav range >= 3% (V50e proposal): {pass_range_3} / {len(df_races)} ({pass_range_3/len(df_races)*100:.0f}%)')


In [ ]:
### CELL 11 - SECTION 7: CONSOLIDATED FINDINGS ###
print('=' * 70)
print('  SECTION 7: Consolidated Findings & Recommendations')
print('=' * 70)

print(f'\n1. LEARNABLE SIGNAL:')
target_std = df_feat['price_return_1step'].std() if 'price_return_1step' in df_feat.columns else 0
print(f'   1-step return std: {target_std:.6f}')
if 'price_return_5step' in df_feat.columns:
    print(f'   5-step return std: {df_feat["price_return_5step"].std():.6f}')

# Check strongest correlations
if 'price_return_5step' in df_feat.columns:
    best_feat = None
    best_corr = 0
    for feat in corr_features:
        if feat in df_feat.columns:
            v = df_feat[[feat, 'price_return_5step']].dropna()
            if len(v) > 100:
                r = abs(v[feat].corr(v['price_return_5step']))
                if r > best_corr:
                    best_corr = r
                    best_feat = feat
    print(f'   Strongest 5-step predictor: {best_feat} (|r|={best_corr:.4f})')

print(f'\n2. CROSS-RUNNER DYNAMICS:')
if corr_samples:
    cs_arr = np.array([c for c in corr_samples if not np.isnan(c)])
    if len(cs_arr) > 0:
        print(f'   Runner vs others correlation: {cs_arr.mean():.4f}')
        print(f'   {"CONFIRMED" if cs_arr.mean() < -0.05 else "WEAK"}: '
              f'{"Overround constraint creates coupled movements" if cs_arr.mean() < -0.05 else "Movements mostly independent"}')
    else:
        print(f'   Insufficient data for cross-runner analysis')

print(f'\n3. OVERROUND:')
if 'overround_start' in df_races.columns:
    print(f'   Median start: {df_races["overround_start"].median():.1f}% -> end: {df_races["overround_end"].median():.1f}%')
    print(f'   Compression is {"real and consistent" if df_races["overround_compression"].median() > 5 else "modest"}')

print(f'\n4. OPPORTUNITY SIZE:')
if 'max_runner_range_pct' in df_races.columns:
    med_range = df_races['max_runner_range_pct'].median()
    print(f'   Median best-runner range: {med_range:.1f}%')
    print(f'   Realistic target PnL per race: ${1000 * med_range/100 * 0.1 * 0.95:.2f} '
          f'(10% of range captured, $1000 capital, 5% commission)')

print(f'\n5. SCALE MISMATCH:')
if scale_features:
    ratio = max(scale_features.values()) / max(min(v for v in scale_features.values() if v > 0), 1e-8)
    print(f'   Feature scale ratio: {ratio:.1f}x')
    print(f'   {"VecNormalize recommended (V50b)" if ratio > 10 else "Manageable without VecNormalize"}')

print(f'\n6. DATA QUALITY:')
print(f'   {loaded} races loaded successfully')
if 'secs_since_last_trade' in df_feat.columns:
    stale_pct = (df_feat['secs_since_last_trade'] > 60).mean() * 100
    print(f'   Stale runner-steps: {stale_pct:.1f}%')

print(f'\n7. STATIONARITY:')
print(f'   Features shift between early and late market — secs_to_off')
print(f'   as a conditioning signal is important (already in V50a).')

print(f'\n8. RECOMMENDATIONS FOR V50:')
print(f'   a) The reward fix (V50a) is the right priority — no feature can')
print(f'      help if the reward teaches the wrong behaviour.')
print(f'   b) VecNormalize (V50b) will help with scale mismatch.')
print(f'   c) Market dynamics features (overround, vol_share) have signal')
print(f'      if cross-runner correlation is negative.')
print(f'   d) Multi-step horizons show stronger signal than single-step —')
print(f'      consistent with swing-trade objective.')

print(f'\n' + '=' * 70)
print(f'  END OF DATA SANITY CHECK')
print(f'=' * 70)
